In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
top3_cities_df = pd.read_csv('/content/drive/MyDrive/820_Project_cleaned_dataset.csv')
top3_cities_df.head()

,review_id,user_id,business_id,stars_review,useful,funny,cool,text,date,name,...,compliment_cool,compliment_funny,compliment_writer,compliment_photos,active_years,elite_count,elite_flag,friends_count,compliment_total,cluster
0,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,Zaika,...,0,0,0,0,11.131507,0,0,1,1,1
1,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31,Dmitri's,...,0,0,0,0,10.723288,0,0,1,1,0
2,l3Wk_mvAog6XANIuGQ9C7Q,ZbqSHbgCjzVAqaa7NKWn5A,EQ-TZ2eeD_E0BHuvoaeG5Q,4,0,0,0,"Locals recommended Milktooth, and it's an amaz...",2015-08-19 14:31:45,Milktooth,...,0,0,0,0,9.556164,0,0,1,0,1
3,8JFGBuHMoiNDyfcxuWNtrA,smOvOajNG0lS4Pq7d8g4JQ,RZtGWDLCAtuipwaZ-UfjmQ,4,0,0,0,Good food--loved the gnocchi with marinara\nth...,2009-10-14 19:57:14,LaScala's,...,1,1,0,0,15.438356,0,0,44,8,3
4,OAhBYw8IQ6wlfw1owXWRWw,1C2lxzUo1Hyye4RFIXly3g,BVndHaLihEYbr76Z0CMEGw,5,0,0,0,"Great place for breakfast! I had the waffle, w...",2014-10-11 16:22:06,Mamas Kitchen,...,0,0,0,0,11.613699,0,0,3,0,3


In [5]:
top3_cities_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1369708 entries, 0 to 1369707
Data columns (total 49 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   review_id           1369708 non-null  object 
 1   user_id             1369708 non-null  object 
 2   business_id         1369708 non-null  object 
 3   stars_review        1369708 non-null  int64  
 4   useful              1369708 non-null  int64  
 5   funny               1369708 non-null  int64  
 6   cool                1369708 non-null  int64  
 7   text                1369708 non-null  object 
 8   date                1369708 non-null  object 
 9   name                1369708 non-null  object 
 10  address             1365077 non-null  object 
 11  city                1369708 non-null  object 
 12  state               1369708 non-null  object 
 13  postal_code         1369523 non-null  float64
 14  latitude            1369708 non-null  float64
 15  longitude      

In [6]:
# deal with na
top3_cities_df = top3_cities_df.dropna(subset=['text'])

# preprocessing
top3_cities_df['address'] = top3_cities_df['address'].fillna('Unknown')
top3_cities_df['postal_code'] = top3_cities_df['postal_code'].astype(str).fillna('Unknown')
top3_cities_df['attributes'] = top3_cities_df['attributes'].fillna('{}')
top3_cities_df['friends'] = top3_cities_df['friends'].fillna('')
top3_cities_df['elite'] = top3_cities_df['elite'].fillna('')
top3_cities_df = top3_cities_df[top3_cities_df['text'].str.strip() != '']

print(top3_cities_df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1369708 entries, 0 to 1369707
Data columns (total 49 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   review_id           1369708 non-null  object 
 1   user_id             1369708 non-null  object 
 2   business_id         1369708 non-null  object 
 3   stars_review        1369708 non-null  int64  
 4   useful              1369708 non-null  int64  
 5   funny               1369708 non-null  int64  
 6   cool                1369708 non-null  int64  
 7   text                1369708 non-null  object 
 8   date                1369708 non-null  object 
 9   name                1369708 non-null  object 
 10  address             1369708 non-null  object 
 11  city                1369708 non-null  object 
 12  state               1369708 non-null  object 
 13  postal_code         1369708 non-null  object 
 14  latitude            1369708 non-null  float64
 15  longitude      

**In this part, we first processed all the parts of the data text that need to be cleaned, so that it will be more convenient to run the model later.**

In [7]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

ps = PorterStemmer()

# Tokenize + Lemmatize
def tokenize_lemmatize(sentence):
    tokens = word_tokenize(sentence)  # Tokenize
    lemmatized_tokens = [ps.stem(word) for word in tokens]  # Lemmatize
    return lemmatized_tokens

#BoW
cv = CountVectorizer(tokenizer=tokenize_lemmatize)
# train model
top3_cities_df_sample = top3_cities_df.sample(frac=0.1, random_state=42)  # 只采样一次
cv.fit(top3_cities_df_sample['text'])
print('number of `tokens`:', len(cv.vocabulary_))
cv.vocabulary_


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


number of `tokens`: 89453


{'thi': 79599,
 'place': 62402,
 'is': 44242,
 'ghetto': 37446,
 '.': 2403,
 'and': 11660,
 'as': 12778,
 'i': 42670,
 'complain': 24202,
 'to': 80372,
 'the': 79407,
 'bouncer': 17450,
 'that': 79384,
 'peopl': 60983,
 'were': 86374,
 'bump': 18793,
 'into': 44050,
 'me': 51379,
 'he': 40515,
 'said': 68988,
 'kept': 46132,
 'bark': 14640,
 'at': 12990,
 'them': 79465,
 'so': 73790,
 'wait-': 85396,
 'they': 79587,
 'are': 12482,
 'allow': 11064,
 'act': 9978,
 'like': 48464,
 'anim': 11778,
 'not': 56652,
 'say': 69959,
 'excus': 32440,
 'feel': 33554,
 'up': 83482,
 'my': 54658,
 'ass': 12908,
 'but': 19132,
 'wait': 85395,
 "'m": 632,
 'forget': 35229,
 'it': 44356,
 'go': 37927,
 'anoth': 11869,
 'classi': 23148,
 'loung': 49376,
 'skip': 73013,
 '!': 0,
 'amaz': 11320,
 'food': 34861,
 '...': 2405,
 'atmospher': 13079,
 '....': 2406,
 'owner': 59359,
 'wonder': 87349,
 'server': 71251,
 'also': 11176,
 'great': 38720,
 'we': 85960,
 'new': 55433,
 'neighborhood': 55260,
 'will': 

**First, I tried to do bow to see if using word frequency alone would produce better results. My guess is that in restaurant reviews, there aren’t many words whose meanings change depending on the context, so bow shouldn’t be a big problem. I chose the clusters obtained in the m1 stage as labels because they had a good f1 score at the time and there were no better labels available.**

**Here, the data crashed several times and it took about half an hour to get the results. The reason is that we had 1370k of data, which was too much for colab to handle, so we sampled 10%.**

In [10]:
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(top3_cities_df_sample, test_size=0.2, random_state=42)

dtmt = cv.transform(df_train['text'])
dtmte = cv.transform(df_test['text'])


classifier = LogisticRegression(max_iter=1000)
classifier.fit(dtmt, df_train['cluster'])

y_pred_bow = classifier.predict(dtmte)
accuracy_bow = accuracy_score(df_test['cluster'], y_pred_bow)
f1_bow = sklearn.metrics.f1_score(df_test['cluster'], y_pred_bow, average='weighted')

print(f"Accuracy (BoW): {accuracy_bow}")
print(f"F1 Score (BoW): {f1_bow}")
print(classification_report(df_test['cluster'], y_pred_bow))

labels = df_test['cluster'].unique()
display(pd.DataFrame(confusion_matrix(df_test['cluster'], y_pred_bow, normalize='true'), columns=labels, index=labels))


Accuracy (BoW): 0.4764373060777514
F1 Score (BoW): 0.4720065960827968
              precision    recall  f1-score   support

           0       0.42      0.29      0.35      2325
           1       0.39      0.33      0.36      5904
           2       0.57      0.54      0.56      8935
           3       0.45      0.56      0.50     10037
           4       0.18      0.11      0.13       189
           5       0.00      0.00      0.00         5

    accuracy                           0.48     27395
   macro avg       0.34      0.30      0.32     27395
weighted avg       0.48      0.48      0.47     27395



/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,1,3,2,0,4,5
1,0.293763,0.168602,0.095914,0.440860,0.000860,0.000000
3,0.044885,0.329438,0.165650,0.458841,0.001186,0.000000
2,0.019922,0.100839,0.537661,0.335534,0.005932,0.000112
0,0.048520,0.168078,0.222278,0.557936,0.003089,0.000100
4,0.010582,0.037037,0.634921,0.211640,0.105820,0.000000
5,0.000000,0.000000,0.600000,0.400000,0.000000,0.000000


**But the results show that bow's data is not reliable, which means that there may be some changes in word meaning that I did not expect. Based on this result, I plan to try TF-IDF again.**

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(tokenizer=tokenize_lemmatize)
tfidf.fit(top3_cities_df_sample['text'])

dtmt_tfidf = tfidf.transform(df_train['text'])
dtmte_tfidf = tfidf.transform(df_test['text'])

print('number of `tokens`:', len(tfidf.vocabulary_))


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


number of `tokens`: 89453


In [12]:
print(df_train.columns)
print(df_test.columns)

Index(['review_id', 'user_id', 'business_id', 'stars_review', 'useful',
       'funny', 'cool', 'text', 'date', 'name', 'address', 'city', 'state',
       'postal_code', 'latitude', 'longitude', 'stars_business',
       'review_count', 'is_open', 'attributes', 'categories', 'hours',
       'name_user', 'review_count_user', 'yelping_since', 'useful_user',
       'funny_user', 'cool_user', 'elite', 'friends', 'fans', 'average_stars',
       'compliment_hot', 'compliment_more', 'compliment_profile',
       'compliment_cute', 'compliment_list', 'compliment_note',
       'compliment_plain', 'compliment_cool', 'compliment_funny',
       'compliment_writer', 'compliment_photos', 'active_years', 'elite_count',
       'elite_flag', 'friends_count', 'compliment_total', 'cluster'],
      dtype='object')
Index(['review_id', 'user_id', 'business_id', 'stars_review', 'useful',
       'funny', 'cool', 'text', 'date', 'name', 'address', 'city', 'state',
       'postal_code', 'latitude', 'longitude', '

**There are a lot of bugs here. I checked the column name to ensure our work is OK.**

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd

classifier_tfidf = LogisticRegression(max_iter=1000)
classifier_tfidf.fit(dtmt_tfidf, df_train['cluster'])

y_pred_tfidf = classifier_tfidf.predict(dtmte_tfidf)

accuracy_tfidf = accuracy_score(df_test['cluster'], y_pred_tfidf)
f1_tfidf = f1_score(df_test['cluster'], y_pred_tfidf, average='weighted')

print(f"Accuracy (TF-IDF): {accuracy_tfidf}")
print(f"F1 Score (TF-IDF): {f1_tfidf}")
print(classification_report(df_test['cluster'], y_pred_tfidf))
display(pd.DataFrame(confusion_matrix(df_test['cluster'], y_pred_tfidf, normalize='true'),
                     columns=classifier_tfidf.classes_, index=classifier_tfidf.classes_))



Accuracy (TF-IDF): 0.5122832633692279
F1 Score (TF-IDF): 0.5025435102012773
              precision    recall  f1-score   support

           0       0.51      0.33      0.40      2325
           1       0.46      0.32      0.38      5904
           2       0.57      0.64      0.60      8935
           3       0.48      0.57      0.52     10037
           4       0.00      0.00      0.00       189
           5       0.00      0.00      0.00         5

    accuracy                           0.51     27395
   macro avg       0.34      0.31      0.32     27395
weighted avg       0.51      0.51      0.50     27395



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,0,1,2,3,4,5
0,0.332043,0.122151,0.106667,0.439140,0.0,0.0
1,0.041159,0.319953,0.202236,0.436653,0.0,0.0
2,0.010520,0.062003,0.637605,0.289871,0.0,0.0
3,0.040749,0.133406,0.260337,0.565508,0.0,0.0
4,0.005291,0.000000,0.830688,0.164021,0.0,0.0
5,0.000000,0.000000,0.800000,0.200000,0.0,0.0


**Although not outstanding, this model clearly performs better than bow, so it is likely that some of the high-frequency words have no real meaning.**

In [14]:
!pip install mglearn

In [15]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
import mglearn
import numpy as np

vectorizer = TfidfVectorizer(max_features=300)
X_tfidf = vectorizer.fit_transform(top3_cities_df_sample["text"])

model = NMF(n_components=5, random_state=42)
model.fit(X_tfidf)

word_importance = model.components_ / model.components_.sum(axis=0, keepdims=True)
feature_names = vectorizer.get_feature_names_out()

mglearn.tools.print_topics(topics=range(5), feature_names=feature_names,
                          sorting=np.argsort(word_importance, axis=1)[:, ::-1], n_words=10,
                           topics_per_chunk=1)


topic 0       
--------      
best          
city          
the           
most          
area          
side          
pork          
beef          
ever          
top           


topic 1       
--------      
was           
ok            
wasn          
tasted        
cooked        
ordered       
overall       
got           
it            
again         


topic 2       
--------      
great         
friendly      
atmosphere    
awesome       
always        
staff         
selection     
love          
amazing       
recommend     


topic 3       
--------      
our           
we            
us            
minutes       
table         
server        
told          
waitress      
she           
her           


topic 4       
--------      
you           
re            
want          
your          
don           
if            
need          
do            
get           
know          




/usr/local/lib/python3.11/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


**Judging from the results, this grouping should be successful. For example, topic 0 is obviously talking about geographical location and things themselves, while topic 1 contains many neutral words, which should be a less extreme conclusion. etc. The final conclusion is that there are several major categories of topics that users like to comment on: "location and food", "neutral thoughts", "good atmosphere", "personal experience", "warning"**

In [17]:
from sklearn.decomposition import PCA
import plotly.express as px
import pandas as pd

topic_embeddings = model.components_

pca = PCA(n_components=3, random_state=42)
topic_embeddings_3d = pca.fit_transform(topic_embeddings)

df_topics = pd.DataFrame(topic_embeddings_3d, columns=['x', 'y', 'z'])
df_topics['topic'] = ["location and food", "neutral thoughts", "good atmosphere", "personal experience", "warning"]

fig = px.scatter_3d(df_topics, x='x', y='y', z='z', text='topic', title="Topic Relationship in 3D")
fig.show()


**After selecting the model with better performance, we performed dimensionality reduction to see if these topics would be significantly close or have other unusual patterns. The results showed that the grouping of this topic was successful, and there was no obvious classification problem.**

In [ ]:
from scipy.cluster.hierarchy import linkage
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import fcluster
topic_scores_test = model.transform(vectorizer.transform(top3_cities_df_sample['text']))
linkage_matrix_topics = linkage(topic_scores_test, method='ward')
sil_topics = silhouette_score(topic_scores_test, fcluster(linkage_matrix_topics, 5, criterion='maxclust'))
print(f"Silhouette Score (Topic Modeling): {sil_topics}")

**This part crashed 4 times and we run out of time to rerun the whole notebook again. The conclusion here should be able to confirm whether our model can be applied to subsequent text analysis.**

**Each crash will take about 50 minutes, and the cause of the crash has never been found. We really can't run the final result.**

**But combined with the above analysis, we can know that this model must be logically correct, and the final uncertainty is just to test how much new data it can cover. When you encounter more reviews in the future, you can use this method to classify user comments and understand their tendencies and dissatisfaction.**